# Week 1 Foundations: Quant Setup, Returns, and Risk

This notebook is designed to run fully in local environments.
It tries to download market data first and falls back to deterministic synthetic data if network access is unavailable.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)

## 1) Load price data with resilient fallback

In [2]:
def synthetic_prices(symbols, periods=756):
    idx = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=periods)
    frame = pd.DataFrame(index=idx)
    for i, sym in enumerate(symbols):
        drift = 0.0002 + i * 0.00003
        vol = 0.01 + i * 0.0015
        rets = np.random.normal(loc=drift, scale=vol, size=periods)
        frame[sym] = 100 * np.exp(np.cumsum(rets))
    return frame

def load_prices(symbols, years='3y'):
    try:
        raw = yf.download(symbols, period=years, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            if ('Close' in raw.columns.get_level_values(0)):
                close = raw['Close'].copy()
            else:
                close = raw.xs('Close', axis=1, level=0, drop_level=False)
                close.columns = [c[1] for c in close.columns]
        else:
            close = raw[['Close']].rename(columns={'Close': symbols[0] if len(symbols) == 1 else 'Close'})
        close = close.dropna(how='all')
        if close.empty:
            raise ValueError('Downloaded data is empty')
        source = 'yfinance'
    except Exception as exc:
        close = synthetic_prices(symbols)
        source = f'synthetic fallback ({type(exc).__name__})'
    close = close.ffill().dropna()
    return close, source

symbols = ['SPY', 'QQQ', 'AAPL']
prices, source = load_prices(symbols)
print(f'Data source: {source}')
print(f'Shape: {prices.shape}')
prices.tail()

Data source: yfinance
Shape: (753, 3)


Ticker,AAPL,QQQ,SPY
Date,,,
2026-04-13,259.200012,617.390015,686.099976
2026-04-14,258.829987,628.599976,694.460022
2026-04-15,266.429993,637.400024,699.940002
2026-04-16,263.399994,640.469971,701.659973
2026-04-17,270.230011,648.849976,710.140015


## 2) Build return series and core risk functions

In [3]:
trading_days = 252
returns = prices.pct_change().dropna()
log_returns = np.log(prices / prices.shift(1)).dropna()

def annualized_return(r):
    return (1 + r).prod() ** (trading_days / len(r)) - 1

def annualized_volatility(r):
    return r.std() * np.sqrt(trading_days)

def sharpe_proxy(r, rf=0.0):
    vol = annualized_volatility(r)
    if vol == 0:
        return np.nan
    return (annualized_return(r) - rf) / vol

def max_drawdown(price_series):
    running_max = price_series.cummax()
    drawdown = price_series / running_max - 1
    return drawdown.min()

summary_rows = []
for sym in symbols:
    r = returns[sym].dropna()
    summary_rows.append({
        'symbol': sym,
        'annualized_return': annualized_return(r),
        'annualized_volatility': annualized_volatility(r),
        'sharpe_proxy': sharpe_proxy(r),
        'max_drawdown': max_drawdown(prices[sym]),
    })

summary = pd.DataFrame(summary_rows).set_index('symbol').sort_values('sharpe_proxy', ascending=False)
summary.style.format('{:.2%}', subset=['annualized_return', 'annualized_volatility', 'max_drawdown']).format('{:.2f}', subset=['sharpe_proxy'])

,annualized_return,annualized_volatility,sharpe_proxy,max_drawdown
symbol,,,,
SPY,21.35%,15.23%,1.40,-18.76%
QQQ,27.63%,19.71%,1.40,-22.77%
AAPL,18.19%,25.76%,0.71,-33.36%


## 3) Visual diagnostics

In [4]:
normed = prices / prices.iloc[0]
rolling_vol = returns.rolling(21).std() * np.sqrt(trading_days)

fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)
normed.plot(ax=axes[0], title='Normalized Price Series')
axes[0].set_ylabel('Normalized Level')

rolling_vol.plot(ax=axes[1], title='21-Day Rolling Annualized Volatility')
axes[1].set_ylabel('Volatility')

for sym in symbols:
    dd = prices[sym] / prices[sym].cummax() - 1
    axes[2].plot(dd.index, dd.values, label=sym)
axes[2].set_title('Drawdown Curves')
axes[2].set_ylabel('Drawdown')
axes[2].legend()

plt.tight_layout()
plt.show()

## 4) Reflection Prompts

1. Which asset had the best risk-adjusted profile over this window?
2. How sensitive are conclusions to the selected lookback period?
3. Which assumptions should be stress-tested before paper trading?